# Wurstpresse
## Skript zur Vorbereitung des Bildimports in iDAI.Objects bzw. Arachne
*Version 2.0 - April 2026*

*von Lukas Lammers für's FAIR.rdm im Spp2143 "Entangled Africa"*

Die Codeblöcke sind jeweils der Reihe nach einzeln auszuführen!
Es gibt vier Abschnitte:
1. In der ersten Zelle müssen die grundlegenden Einstellungen korrekt ausgefüllt werden.
2. Als nächstes werden die Stände der laufenden Bildnummern abgeglichen und eingelesen.
3. Die Bilder werden umbenannt.
4. Die Ordnerbäume werden erstellt und eine ZIP-Datei im Zielverzeichnis ausgegeben.

Die ZIP-Datei eignet sich für einen Uplaod in HacVia: [https://images.arachne.dainst.org](https://images.arachne.dainst.org/) bzw. [https://images.arachne.test.dainst.org](https://images.arachne.test.dainst.org/)


## 1. Einstellungen
### 1.1 Variablen und Parameter
Es gibt das Quellverzeichnis, in dem die zu verarbeitenden Bilder liegen. Das Zielverzeichnis ist der Pfad, wo die Ordnerbäume zum Import erstellt werden sollen und in den die umbenannten Bilder einsortiert werden. Die ZIP-Datei für den Upload in HacVia wird ebenfalls im Zielverzeichnis generiert.

Im Arbeitsverzeichnis muss eine Tabelle mit einer Konkordanz aus Bildname und Entity-ID des zugehörigen Objekts liegen. Die Spaltennamen dieser Bildtabelle mit Entity ID und Bildnamen müssen separat angegeben werden.

Projektdaten sind relevant für die Ermittlung der laufenden Nummer sowie die Bezeichnung der Batch in HacVia.

In [ ]:
#Hier die Verzeichnispfade eingeben, wo die umzubennenenden Bilder liegen und wo sie hin sollen
quellverzeichnis = "D:\\Eigene Dateien\\Dokumente\\DCH\\Wurstpresse_2\\Bilder"
zielverzeichnis = "D:\\Eigene Dateien\\Dokumente\\DCH\\Wurstpresse_2\\Testziel_6"

#Hier den Verzeichnispfad des Arbeitsordners angeben.
arbeitsverzeichnis = "D:\\Eigene Dateien\\Dokumente\\DCH\\Wurstpresse_2"

#Hier die Dateinamen der Bildtabelle mit Spaltennamen eingeben.
bildtabelle = "Testbilder.xlsx"
spaltenname_entity_id = "Entity ID"
spaltenname_bildnamen = "Bildnamen"

#Projektdaten
project = 'P01'
batchname = "SPP2143_Bildimport_Test_6"
optional_sofs_drive_letter = 'Z:'

### 1.2 Import Modules
Die Bibliotheken im folgenden Codeblock werden für das Skript benötigt.
Sie können einfach über die Konsole bspw. mit `pip install pandas` installiert werden.
Python mit pip wird dazu benötigt.

In [ ]:
import sys
import numpy as np
import pandas as pd
import os
import csv
import requests

##### Check 1 - Module importiert?

In [ ]:
if "numpy" in sys.modules and "pandas" in sys.modules and "os" in sys.modules and "csv" in sys.modules:
    print("Check 1 erfolgreich.")
else:
    print("Fehler beim Import der Module. Arbeit abbrechen und Skript prüfen.")

##### Check 2 - Pfade korrekt angegeben?

In [ ]:
if os.path.exists(quellverzeichnis):
    print("Quellverzeichnispfad OK.")
else:
    print("Quellverzeichnis nicht gefunden! Schreibweise oder Anzahl der Slashes überprüfen!")
if os.path.exists(zielverzeichnis):
    print("Zielverzeichnispfad OK.")
else:
    print("Zielverzeichnis nicht gefunden! Schreibweise oder Anzahl der Slashes überprüfen!")
if os.path.exists(arbeitsverzeichnis):
    print("Arbeitsverzeichnispfad OK.")
else:
    print("Arbeitsverzeichnis nicht gefunden! Schreibweise oder Anzahl der Slashes überprüfen!")
if os.path.exists(f"{arbeitsverzeichnis}\\{bildtabelle}"):
    print("Bildtabelle OK.")
else:
    print("Bildtabelle nicht gefunden! Schreibweise oder Anzahl der Slashes überprüfen!")

## 2. Stand Einlesen und Abgleichen
### 2.1 Entity IDs einlesen

In [ ]:
#get entityIDs
if ".csv" in bildtabelle:
    df_topos = pd.read_csv(f'{arbeitsverzeichnis}\\{bildtabelle}', encoding='utf-8', sep=';')
elif ".xlsx" in bildtabelle:
    df_topos = pd.read_excel(f'{arbeitsverzeichnis}\\{bildtabelle}', engine='openpyxl')
else:
    print("Dateiformat der Bildtabelle nicht erkannt! Bitte .csv oder .xlsx verwenden und Schreibweise überprüfen!")

#List from column id
eids = df_topos[f'{spaltenname_entity_id}'].tolist()

#all entries in the list have to be string

for i in range(len(eids)):
    eids[i] = str(eids[i])

print(eids)

### 2.2 Ableich über Arachnes REST API

In [ ]:
#Find out the letter of the drive where this file is located or set it manually above
drive_letter = os.path.splitdrive(os.getcwd())[0]

if optional_sofs_drive_letter:
    drive_letter = optional_sofs_drive_letter

print(drive_letter)

path = f"{drive_letter}\\FAIR.rdm AAArC 5\\Empfang\\Importe\\{project}\\{project}_Bildimport"

if not os.path.exists(path):
    print(f"Directory {path} does not exist. Please check the path and try again.")

#check if a file with "StdLfdBildNr" already exists in the directory
existing_files = [f for f in os.listdir(path) if "StdLfdBildNr" in f]
if existing_files:
    recent_file = existing_files[-1]  # Assuming the most recent file is the last one in the list
    df_count_sofs = pd.read_csv(os.path.join(path, recent_file), sep=';', encoding='utf-8')
    print(f"Existing file found: {recent_file}. It will be used for comparison.")
    counts_dict_sofs = {str(k): v for k, v in zip(df_count_sofs['Entity ID'], df_count_sofs['Count'])}
else:
    print("No existing file with 'StdLfdBildNr' found. A new file will be created.")

print(counts_dict_sofs)

def get_metadata(id):
    '''Get metadata for a given entity ID from the Arachne API.'''
    arachne_api = "https://arachne.dainst.org/data/entity/"
    data = {}
    entity_id = id  # Use the provided ID
    url = arachne_api + entity_id
    response = requests.get(url)
    data = response.json()
    return data

def extract_count(imagenames):
    '''Find the maximum count from the image subtitles.'''
    maxcount = 0
    for imagename in imagenames:
        if "," in imagename:
            count = imagename.split(",")[-1]
            count = int(count)
            if count > maxcount:
                maxcount = count
    return maxcount

#Main Loop

counts_dict_api = {}

for id in eids:
    data = get_metadata(id)
    if "images" not in data:
        print(f"Entity ID: {id} has no images.")
        counts_dict_api[id] = 0
        continue
    else:
        images = data['images']
        imagenames = []
        for image in images:
            imagenames.append(image['imageSubtitle'])
        count = extract_count(imagenames)
        print(f"Entity ID: {id}, Count: {count}")
        counts_dict_api[id] = count

print(counts_dict_api)

In [ ]:
#Get all keys from counts_dict_api and counts_dict_sofs
keys_api = set(counts_dict_api.keys())
keys_sofs = set(map(str, counts_dict_sofs.keys()))
#combine them in one set
all_keys = keys_api.union(keys_sofs)
#to list
all_keys = list(all_keys)

print(all_keys)
print(counts_dict_api)
print(counts_dict_sofs)

current_counts = {}

for key in all_keys:
    print(f"Processing Entity ID: {key}")
    if key in counts_dict_api and key in counts_dict_sofs:
        count_api = counts_dict_api[key]
        count_sofs = counts_dict_sofs[key]
        if count_api > count_sofs:
            print(f"{key} has a higher count in API: {count_api} > {count_sofs}")
            count_current = count_api
        else:
            count_current = count_sofs
    elif key in counts_dict_api:
        count_current = counts_dict_api[key]
    elif key in counts_dict_sofs:
        count_current = counts_dict_sofs[key]
    else:
        count_current = 0
    current_counts[key] = count_current

print(current_counts)

### 2.3 Speichern des aktuellen Stands

In [ ]:
#safe counts as csv
counts_df = pd.DataFrame(list(current_counts.items()), columns=['Entity ID', 'Count'])
counts_df.to_csv(f'{path}//{project}_StdLfdBildNr.csv', index=False, encoding='utf-8', sep=';')

## 3. Umbenennen der Bilder

### 3.1 Zuordnung Bildname und Entity ID

In [ ]:
#dict with bildnamen as key and entity id as value
konkordanz_dict = dict(zip(df_topos[f'{spaltenname_bildnamen}'], df_topos[f'{spaltenname_entity_id}']))
print(konkordanz_dict)
print(eids)

### 3.2 Renamedict erzeugen:
Das "Renamedict" ist eine Konkordanz zwischen altem Bildnamen und neuem Bildnamen.

In [ ]:
renamedict = {}

for bild in konkordanz_dict.keys():
    alter_bildname = bild.removesuffix(".tif")   # Python 3.9+
    entity_id = konkordanz_dict[bild]
    entity_id = str(entity_id)
    if entity_id in current_counts:
        newcount = (current_counts.get(entity_id))+ 1
        current_counts.update({entity_id:newcount})
        if len(str(newcount))==1:
            newname = f"{alter_bildname}_{entity_id},00{newcount}.tif"
        elif len(str(newcount))==2:
            newname = f"{alter_bildname}_{entity_id},0{newcount}.tif"
        else:
            newname = f"{alter_bildname}_{entity_id},{newcount}.tif"
    else:
        current_counts[entity_id] = 3
        startcount = 3
        newname = f"{alter_bildname}_{entity_id},00{startcount}.tif"
        print(f"Neue Entity-ID {entity_id} im aktuellen Stand angelegt!")
    renamedict[bild]=newname
#print zu Kontrolle
print(current_counts)
print(renamedict)

### 3.4 Aktualiserung der laufenden Nummer

In [ ]:
#Save current_counts again with the new counts after renaming
counts_df_updated = pd.DataFrame(list(current_counts.items()), columns=['Entity ID', 'Count'])
counts_df_updated.to_csv(f'{path}//{project}_StdLfdBildNr.csv', index=False, encoding='utf-8', sep=';')

### 3.5 Dateifinder aktivieren
Diese Funktion durchsucht den Verzeichnispfad.

In [ ]:
def find_all(name, path):
    result = []
    for root, dirs, files in os.walk(path):
        if name in files:
            result.append(os.path.join(root))
        #else: print("Name is invalid!")
    return result
#Check 6 - find_all aktiviert?

### 3.6 Umbenennen

In [ ]:
#Diese Variablen sind für die Anzeige des Umbennenungsfortschritts in Prozent
x=0
gesamtzahl = len(renamedict)

#Dieser Loop bennent die Bilder um
for bildnamen in renamedict:
    newname = renamedict.get(bildnamen)
    pfad_list = find_all(bildnamen, quellverzeichnis)
    pfad = "".join(pfad_list)
    src =f"{pfad}/{bildnamen}"
    dst =f"{pfad}/{newname}"
    if os.path.isfile(src):
        os.rename(src, dst)
        print (f"{bildnamen} wurde erfolgreich umbenannt.")
        x = x+1
    else:
        print (f" {bildnamen} existiert nicht!")
    xfortschritt = (x*100)/gesamtzahl
    xfortschritt = round(xfortschritt, 2)
    print(f"{xfortschritt}% abgeschlossen")

## 4. Verschieben und Packen
### 4.1 Ordner erstellen

In [ ]:
eids_set = set(eids)
eids_set_list = list(eids_set)

#Überornder nach batchname erstellen

if not os.path.exists(f"{zielverzeichnis}/{batchname}"):
    os.makedirs(f"{zielverzeichnis}/{batchname}")

#Verzeichnisse nach Entity erstellen

for nummer in eids_set_list:
    newfolder = f"{zielverzeichnis}/{batchname}/{nummer}"
    datenbankfertig = f"{zielverzeichnis}/{batchname}/{nummer}/edited"
    rohscans =  f"{zielverzeichnis}/{batchname}/{nummer}/unedited"
    if not os.path.exists(newfolder):
        os.makedirs(newfolder)
    if not os.path.exists(datenbankfertig):
        os.makedirs(datenbankfertig)
    if not os.path.exists(rohscans):
        os.makedirs(rohscans)

### 4.2 Bilder in Ordner verschieben

In [ ]:
#Alle Dateien finden und verschieben
import shutil

#Diese Variablen sind für die Anzeige des Verschiebungsfortschritts in Prozent
y=0
gesamtzahl = len(bildnamen)

for bild in renamedict.values():
    pfad_list = find_all(bild, quellverzeichnis)
    pfad = "".join(pfad_list)
    source = f"{pfad}/{bild}"
    if os.path.isfile(source):
        aaarc_fileending = bild.split("_")[1]
        archneentityid = aaarc_fileending.split(",")[0]
        destination = f"{zielverzeichnis}/{batchname}/{archneentityid}/unedited"
        test = f"{destination}/{bild}"
        if os.path.isfile(test):
            print(f"Das Bild {bild} wurde bereits verschoben!")
        else:
            shutil.move(source, destination)
            print (f"{bild} wurde erfolgreich verschoben.")
            y = y+1
    else:
        print (f" {bild} existiert nicht!")
    yfortschritt = (y*100)/gesamtzahl
    yfortschritt = round(yfortschritt, 2)
    print(f"{yfortschritt}% abgeschlossen")

### 4.3 In ZIP verpacken
HacVia akzeptiert ausschließlich ZIP-Datein im Bulk-Upload.

In [ ]:
#Create a ZIP file out of Zielverzeichnis

shutil.make_archive(f"{zielverzeichnis}", 'zip', f"{zielverzeichnis}")

#ZIP file has to have the same name as batchname for the upload to work, so we rename it here
shutil.move(f"{zielverzeichnis}.zip", f"{zielverzeichnis}/{batchname}.zip")
